# 09 · Heimdall's local lane – a model on your own machine
The gateway can also front a model that runs **on the demo laptop**. Ollama serves it, and an **ngrok** tunnel makes it reachable for APIM.
Clients keep using the same endpoint and the same key. The only difference is the `model` name: if it is listed in the named value `ollama-models`, the router sends the call to the `asgard-ollama` backend instead of Foundry.

```
client --api-key--> APIM /bifrost/v1 --(model in ollama-models?)--> asgard-ollama --basic auth--> ngrok tunnel --> Ollama :11434
                                     \--(otherwise)----------------> Foundry (managed identity)
```

**Prereq:** deploy with the local lane enabled (README → *Local Ollama via ngrok*):
`azd env set HEIMDAL_ENABLE_OLLAMA true`, `azd env set HEIMDAL_OLLAMA_MODEL <model>`, then `azd up`.
The azd `preprovision` hook (`scripts/ollama_tunnel.py`) starts Ollama and ngrok and wires the tunnel URL into APIM.
Run this notebook **on the machine that hosts Ollama**: the status check below looks at the local processes.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
import os
sys.path.insert(0, '../scripts')
import ollama_tunnel  # same checks the azd hook runs (read-only here)

LOCAL_MODELS = [m for m in os.getenv('HEIMDAL_OLLAMA_MODELS', '').split(',') if m]
assert LOCAL_MODELS, 'HEIMDAL_OLLAMA_MODELS is empty: deploy with HEIMDAL_ENABLE_OLLAMA=true (see README) and re-run.'
LOCAL = LOCAL_MODELS[0]

served = ollama_tunnel.ollama_models() or []
TUNNEL = ollama_tunnel.find_tunnel()
pd.DataFrame([
    {'check': '🦙 Ollama serving on :11434', 'status': bool(served)},
    {'check': f'📦 model {LOCAL} pulled', 'status': ollama_tunnel.has_model(served, LOCAL)},
    {'check': '🚇 ngrok tunnel to :11434', 'status': TUNNEL or False},
])

> ⚠️ If a check is `False`, run `python scripts/ollama_tunnel.py` (with `HEIMDAL_ENABLE_OLLAMA=true`) or `azd provision`.
> If ngrok restarted with a **new URL**, run `azd provision` again so the APIM backend follows it.

### 🌈 First call through the local lane
Same `ask()` as in notebook 01, just with the local model name. `x-heimdal-route-reason` starts with `local-ollama;`.
Small reasoning models spend tokens on thinking first, so give them some room with `max_tokens`.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
r = ask('In one sentence, who is Heimdall in Marvel?', user='thor', model=LOCAL, max_tokens=800)
print(r['answer'])
print({k: r[k] for k in ('user', 'tier', 'model', 'reason', 'remaining_tokens', 'prompt_tokens', 'completion_tokens')})

### ⚖️ Local vs Foundry, side by side
The same prompt goes to the local model and to the router (`heimdal-auto` → Foundry). Both go through the same gateway, key and token budget.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
import time

prompt = 'Name three Asgardians and their weapons. Answer as a short list.'
rows = []
for label, model in [('🦙 local', LOCAL), ('☁️ foundry', 'heimdal-auto')]:
    t0 = time.perf_counter()
    r = ask(prompt, user='thor', model=model, max_tokens=800)
    rows.append({'prompt': label, 'seconds': round(time.perf_counter() - t0, 1), **r})
show(rows, cols=('model', 'tier', 'seconds', 'prompt_tokens', 'completion_tokens', 'answer'))

### 🌊 Streaming
Streaming works through APIM and the tunnel as well. `stream_options={"include_usage": True}` makes the last chunk carry the token usage, which the gateway's `llm-emit-token-metric` needs.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
stream = client('thor').chat.completions.create(
    model=LOCAL, stream=True, stream_options={'include_usage': True}, max_tokens=800,
    messages=[{'role': 'user', 'content': 'Count from one to five in words.'}])
usage = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='', flush=True)
    usage = chunk.usage or usage
print('\n\nusage:', usage)

### 🔒 The tunnel only opens for the gateway
The ngrok URL is public, but the tunnel runs with an ngrok traffic policy that requires **basic auth**.
Only APIM knows the credentials, from the secret named value `ollama-basic-auth`. A direct call is rejected, and so is a call that carries a user's APIM key.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
import requests

if not TUNNEL:
    print('No local tunnel found - start it first (see the status check above).')
else:
    anon = requests.get(f'{TUNNEL}/v1/models', timeout=15)
    with_apim_key = requests.get(f'{TUNNEL}/v1/models', headers={'api-key': user_key('thor')}, timeout=15)
    print("🚫 direct, no credentials  :", anon.status_code)
    print("🚫 direct, thor's APIM key :", with_apim_key.status_code)
    print("✅ via the gateway         :", ask('Say OK.', user='thor', model=LOCAL, max_tokens=400)['model'])

### 🐜 Send a whole tier to the laptop
The Ollama check runs on the **routed** model. So if you point a tier's named value at the local model, the router sends that tier to Ollama, with no client change.
Here `model-tier-pym` (short, simple prompts) goes local for a moment and is then restored from `infra/config/heimdal.json`.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
import time

set_router(model_tier_pym=LOCAL)
try:
    for _ in range(12):  # named values take up to ~1 minute to reach the gateway
        r = ask('Hi Heimdall!', user='hulk', max_tokens=400)
        if r['model'] == LOCAL:
            break
        print(f"still {r['model']} - waiting for the named value...")
        time.sleep(10)
    display(show([{'prompt': 'heimdal-auto, short', **r}]))
finally:
    set_router(model_tier_pym=CONFIG['tierModels']['pym'])

### 👁️ The All-Seeing Eye sees the laptop too
Local calls show up in the same token metrics (dimension `Model` = the Ollama model) and in the LLM gateway log, next to the Foundry deployments.
Ingestion takes a few minutes; `kql_wait()` retries.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *
df = kql_wait(KQL_TOKENS_BY_USER_METRIC, hours=3)
df[df['Model'].isin(LOCAL_MODELS)] if len(df) else df

### 🪵 Which backend answered? The gateway log knows
Every call writes a row to `ApiManagementGatewayLogs` that includes the **backend** APIM forwarded it to (`BackendId` = `asgard-ollama` for the laptop).
`KQL_CALLS_BY_BACKEND` joins it with the LLM gateway log (tokens, model) and splits all Bifrost calls into local Ollama and Foundry.
The Azure Monitor workbook **Heimdal - LLM token usage** has the same view in its *Local Ollama via ngrok* section, with latency, failures and the latest local calls.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

df = kql_wait(KQL_CALLS_BY_BACKEND, hours=3)
df